In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    auc,
    f1_score,
    recall_score,
    accuracy_score,
    classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings("ignore")

# Try to import boosting libraries
_xgb_available, _lgb_available = True, True
try:
    import xgboost as xgb
except ImportError:
    _xgb_available = False
    print(" XGBoost not installed — skipping those models.")
try:
    import lightgbm as lgb
except ImportError:
    _lgb_available = False
    print(" LightGBM not installed — skipping those models.")


## Paths
INPUT_TRAIN_PATH = Path("../data/train_X.csv")
INPUT_TEST_PATH = Path("../data/test_X.csv")
INPUT_TRAIN_Y_PATH = Path("../data/train_y.csv")
INPUT_TEST_Y_PATH = Path("../data/test_y.csv")
#
# -------------------------------
# Load dataset
# -------------------------------
X_train = pd.read_csv(INPUT_TRAIN_PATH)
print("head of X_train:", X_train.head)
#
y_train = pd.read_csv(INPUT_TRAIN_Y_PATH)
print("head of y_train:", y_train.head)
#
X_test = pd.read_csv(INPUT_TEST_PATH)
print("head of X_test:", X_test.head)
#
y_test = pd.read_csv(INPUT_TEST_Y_PATH)
print("head of y_test:", y_test.head)


# --- Helper metric function ---
def pr_auc(y_true, y_scores):
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    return auc(recall, precision)

# --- Evaluate function ---
def evaluate_model(model, X_train, y_train, X_test, y_test, use_smote=False):
    """
    Fits model (optionally with SMOTE) and computes key metrics on test set.
    """
    steps = []
    steps.append(("scaler", StandardScaler()))
    if use_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    steps.append(("clf", model))
    
    pipeline = ImbPipeline(steps=steps) if use_smote else Pipeline(steps=steps)
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_score = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else y_pred
    
    metrics = {
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": pr_auc(y_test, y_score),
        "f1": f1_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
    }
    return metrics, classification_report(y_test, y_pred, digits=4)

# --- Main Evaluation Runner ---
def run_all_models(X_train, y_train, X_test, y_test, random_state=42):
    results = {}

    print("\n Logistic Regression (class_weight='balanced')")
    logit = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000, random_state=random_state)
    results["LogisticRegression_balanced"], rep = evaluate_model(logit, X_train, y_train, X_test, y_test)
    print(rep)

    print("\n Random Forest (baseline)")
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=random_state)
    results["RandomForest"], rep = evaluate_model(rf, X_train, y_train, X_test, y_test)
    print(rep)

    # --- Boosting Models ---
    pos = np.sum(y_train == 1)
    neg = np.sum(y_train == 0)
    scale_pos_weight = neg / pos if pos > 0 else 1.0

    if _xgb_available:
        print("\n XGBoost (scale_pos_weight)")
        xgb_clf = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            random_state=random_state,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            use_label_encoder=False,
        )
        results["XGBoost_weighted"], rep = evaluate_model(xgb_clf, X_train, y_train, X_test, y_test)
        print(rep)

    if _lgb_available:
        print("\n LightGBM (is_unbalance=True)")
        lgb_clf = lgb.LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            is_unbalance=True,
            force_row_wise=True,
            verbosity=-1,
            random_state=random_state,
        )
        results["LightGBM_unbalanced"], rep = evaluate_model(lgb_clf, X_train, y_train, X_test, y_test)
        print(rep)

    # --- SMOTE + Random Forest ---
    print("\n🔁 SMOTE + Random Forest")
    smote_rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=random_state)
    results["SMOTE_RandomForest"], rep = evaluate_model(smote_rf, X_train, y_train, X_test, y_test, use_smote=True)
    print(rep)

    # --- SMOTE + XGBoost ---
    if _xgb_available:
        print("\n🔁 SMOTE + XGBoost")
        smote_xgb = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            random_state=random_state,
            eval_metric="logloss",
            use_label_encoder=False,
        )
        results["SMOTE_XGBoost"], rep = evaluate_model(smote_xgb, X_train, y_train, X_test, y_test, use_smote=True)
        print(rep)

    # --- Summary Table ---
    df_summary = pd.DataFrame([
        {"Model": k, **v} for k, v in results.items()
    ]).sort_values(by="pr_auc", ascending=False).reset_index(drop=True)

    print("\n📊 Summary (sorted by PR-AUC):")
    print(df_summary.round(4))
    return df_summary

# Example usage:
df_summary = run_all_models(X_train, np.ravel(y_train), X_test, np.ravel(y_test))


head of X_train: <bound method NDFrame.head of        perc_premium_paid_by_cash_credit    Income  premium_to_income  \
0                             -0.762252 -1.056313          -0.441069   
1                             -0.851828  0.619890          -0.985752   
2                             -0.481582  0.030873           0.135439   
3                             -0.860785  0.166194           1.061873   
4                              0.040943 -0.099648          -0.110840   
...                                 ...       ...                ...   
63877                          1.333818 -0.459054          -0.645000   
63878                         -0.577129  0.496163           0.912571   
63879                         -0.287501  0.678221          -1.010833   
63880                         -0.938418  0.173893           1.049436   
63881                         -0.935432 -0.613486          -0.368195   

       application_underwriting_score  age_in_days  premiums_paid_ratio  \
0            